In [54]:
# class Task, class Core, class Stage and getMinTime function
class Task:
    def __init__(self, s, i):
        self.id = i
        self.name = "Task-" + str(s.getId()) + "-" + str(i)
        self.stage = s
        self.remainingTime = self.stage.getAveTaskTime()
        self.startTime = 0.0
        self.endTime = 0.0
    
    def getName(self):
        return self.name
    
    def getTime(self):
        return self.remainingTime
    
    def getId(self):
        return self.id
    
    def decrementTime(self, amount):
        self.remainingTime = self.remainingTime - amount
        
    def setCompleted(self, t):
        self.remainingTime = 0.0
        self.stage.incrNumTasksCompleted()
        self.endTime = t
        self.stage.setEndTime(t, self)
        
    def setStartTime(self, t): 
        self.startTime = t; 
        self.stage.setStartTime(t, self)

class Core:
    def __init__(self, i):
        self.id = "Core-" + str(i)
        self.executingTask = None
    def isFree(self):
        if (self.executingTask == None):
            return True
        else:
            return self.executingTask.getTime() <= 0.00000001
    def setBusy(self, t, time):
        self.executingTask = t
        t.setStartTime(time)

class Stage:
    def __init__(self, i, numTasks, t):
        self.id = i
        self.numberOfTasks = numTasks
        self.aveTaskTime = t
        self.predecessors = []
        self.successors = []
        self.numOfTasksCompleted = 0
        self.nextTaskId = self.numOfTasksCompleted
        self.startTime = 0.0
        self.endTime = 0.0
    
    def isCompleted(self):
        return self.numOfTasksCompleted == self.numberOfTasks
    
    def getNumberOfTasks(self):
        return self.numberOfTasks
    
    def getNumberOfTasksCompleted(self):
        return self.numOfTasksCompleted
    
    def getIdOfNextTaskToAllocate(self):
        self.nextTaskId = self.nextTaskId+1
        if (self.nextTaskId >= self.numberOfTasks):
            self.nextTaskId = self.numberOfTasks
        return self.nextTaskId
                                  
    def getAveTaskTime(self):
        return self.aveTaskTime 
                                  
    def getId(self):
        return self.id
    
    def addPredecessor(self, s):
        self.predecessors.append(s)
        s.addSuccessor(self)
   
    def addSuccessor(self, s):
        self.successors.append(s)                             
    
    def getPredecessors(self):
        return self.predecessors
                                  
    def getSuccessors(self):
        return self.successors
    
    def incrNumTasksCompleted(self):
        self.numOfTasksCompleted = self.numOfTasksCompleted + 1                
    
    def createTasks(self):
        list = []
        for i in range(1, self.numberOfTasks+1):
            list.append(Task(self, i))
        return list
                                  
    def getCompletionTime(self):
        return self.endTime-self.startTime
                                  
    def setStartTime(self, time, t):
        if(t.getId() == 1):
            #print("stage" + str(self.id) + " startTime: " + str(time))
            self.startTime = time
    
    def setEndTime(self, time, t):
        if(t.getId() == self.numberOfTasks):
            #print("stage" + str(self.id) + " endTime: " + str(time))
            self.endTime = time                               

def getMinTime(list):
    m = 1000000000.0
    for t in list:
        if(t.getTime() < m):
            m = t.getTime()
    return m

In [55]:
stages = []
c1 = Core(1)
c2 = Core(2)
c3 = Core(3)
c4 = Core(4)
c5 = Core(5)
c6 = Core(6)
c7 = Core(7)
c8 = Core(8)

readyStages = []
executingTasks = []
readyTasks = []
cores = []

cores.append(c1)
cores.append(c2)
cores.append(c3)
cores.append(c4)
cores.append(c5)
cores.append(c6)
cores.append(c7)
cores.append(c8)

In [56]:

# Define stages: Stage(stage_id, num_tasks, avg_task_time)

stage0 = Stage(0, 1, 10.5)
stages.append(stage0)

stage1 = Stage(1, 1, 1.875)
stages.append(stage1)

stage2 = Stage(2, 15, 1.625)
stages.append(stage2)

stage3 = Stage(3, 40, 2.0)
stages.append(stage3)

stage4 = Stage(4, 1, 1.0)
stages.append(stage4)

stage5 = Stage(5, 1, 1.0)
stages.append(stage5)



stage1.addPredecessor(stage0)
stage2.addPredecessor(stage0)

stage3.addPredecessor(stage1)
stage3.addPredecessor(stage2)

stage4.addPredecessor(stage3)

stage5.addPredecessor(stage4)

In [58]:
# measured DAM experiment table
dam_cases = [
    (500, 5, 15, 56),
    (500, 4, 12, 55),
    (500, 12, 8, 51),
    (500, 3, 8, 57),
    (500, 10, 6, 53)
]

In [59]:
def run_dam(idleTime, backlogTime):
    # recreate stages every run
    stage0 = Stage(0, 1, 10.5)
    stage1 = Stage(1, 1, 1.875)
    stage2 = Stage(2, 15, 1.625)
    stage3 = Stage(3, 40, 2.0)
    stage4 = Stage(4, 1, 1.0)
    stage5 = Stage(5, 1, 1.0)

    stages = [stage0, stage1, stage2, stage3, stage4, stage5]

    stage1.addPredecessor(stage0)
    stage2.addPredecessor(stage0)
    stage3.addPredecessor(stage1)
    stage3.addPredecessor(stage2)
    stage4.addPredecessor(stage3)
    stage5.addPredecessor(stage4)

    readyStages = []
    readyTasks = []

    for s in stages:
        if len(s.getPredecessors()) == 0:
            readyStages.append(s)
            for t in s.createTasks():
                readyTasks.append(t)

    cores = [Core(i) for i in range(4)]

    Ttotal = 0.0
    executingTasks = []
    backlogTimer = 0.0
    coreIdleTime = {c: 0.0 for c in cores}

    while len(readyStages) > 0:

        index = 0
        i = 0

        while i < len(cores):
            c = cores[i]

            if c.isFree():

                if len(readyTasks) == 0:
                    break

                if index == len(readyStages):
                    index = 0

                s = readyStages[index]

                # find next available task from this stage
                taskIndex = -1
                for j in range(len(readyTasks)):
                    if readyTasks[j].getName().startswith("Task-" + str(s.getId()) + "-"):
                        taskIndex = j
                        break

                if taskIndex != -1:
                    t = readyTasks[taskIndex]
                    del readyTasks[taskIndex]

                    c.setBusy(t, Ttotal)
                    executingTasks.append(t)
                    coreIdleTime[c] = 0.0

                index += 1
                i += 1

            else:
                i += 1

        if len(executingTasks) == 0:
            break

        Telapse = getMinTime(executingTasks)
        Ttotal += Telapse

        allBusy = True
        for c in cores:
            if c.isFree():
                allBusy = False
                break

        if len(readyTasks) > 0 and allBusy:
            backlogTimer += Telapse
        else:
            backlogTimer = 0.0

        # add core if backlog persists
        if len(readyTasks) > 0 and allBusy and backlogTimer >= backlogTime:
            newCore = Core(len(cores))
            cores.append(newCore)
            coreIdleTime[newCore] = 0.0
            backlogTimer = 0.0

        notCompletedTasks = []

        for t in executingTasks:
            if t.getTime() > Telapse:
                t.decrementTime(Telapse)
                notCompletedTasks.append(t)
            else:
                t.setCompleted(Ttotal)

                for c in cores:
                    if c.executingTask == t:
                        c.executingTask = None
                        break

        executingTasks = notCompletedTasks

        # update idle times
        for c in cores:
            if c.isFree():
                coreIdleTime[c] += Telapse
            else:
                coreIdleTime[c] = 0.0

        # remove idle cores, but keep at least 4 initial cores
        updatedCores = []

        for c in cores:
            if c.isFree() and coreIdleTime[c] >= idleTime and len(cores) > 4:
                continue
            else:
                updatedCores.append(c)

        cores = updatedCores

        completedStages = []
        incompleteStages = []

        for s in readyStages:
            if s.isCompleted():
                completedStages.append(s)
            else:
                incompleteStages.append(s)

        readyStages = incompleteStages

        for s in completedStages:
            for s2 in s.getSuccessors():
                if all(p.isCompleted() for p in s2.getPredecessors()):
                    readyStages.append(s2)
                    for t in s2.createTasks():
                        readyTasks.append(t)

    warmUpAndStageTransferTime = 17.12
    return Ttotal + warmUpAndStageTransferTime

In [60]:
print("Data | Idle | Backlog | Measured | Predicted | Error (%)")
print("-" * 65)

total_error = 0

for datasize, idle, backlog, measured in dam_cases:
    predicted = run_dam(idle, backlog)
    error = abs(predicted - measured) / measured * 100
    total_error += error

    print(datasize, idle, backlog, measured, round(predicted, 2), round(error, 2))

avg_error = total_error / len(dam_cases)

print("\nAverage Error:", round(avg_error, 2), "%")

Data | Idle | Backlog | Measured | Predicted | Error (%)
-----------------------------------------------------------------
500 5 15 56 57.75 3.12
500 4 12 55 57.75 4.99
500 12 8 51 55.75 9.3
500 3 8 57 55.75 2.2
500 10 6 53 55.75 5.18

Average Error: 4.96 %
